In [1]:
%env HF_ENDPOINT=https://hf-mirror.com
%env HF_HOME=/root/autodl-tmp/hf

env: HF_ENDPOINT=https://hf-mirror.com
env: HF_HOME=/root/autodl-tmp/hf


In [2]:
# 加载model和Tokenizer
from transformers import AutoModelForCausalLM,AutoTokenizer
import torch

model_name = 'Qwen/Qwen3-8B'
model = AutoModelForCausalLM.from_pretrained(model_name,dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [3]:
# 数据集
from datasets import load_dataset
dataset_dict = load_dataset('json',data_files={"train":"data/keywords_data_train.jsonl","test":"data/keywords_data_test.jsonl"})

def map_func(example):
    conversation = example['conversation']
    messages = []
    for item in conversation:
        messages.append({'role':'user','content':item['human']})
        messages.append({'role':'assistant','content':item['assistant']})
    return {'messages':messages}

dataset_dict = dataset_dict.map(map_func,batched=False,remove_columns=['dataset','conversation','category','conversation_id'])

In [4]:
from trl import SFTConfig,SFTTrainer
from peft import LoraConfig

# TODO: Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
rank_dimension = 4
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_alpha = 8
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,  # Rank dimension - typically between 4-32
    lora_alpha=lora_alpha,  # LoRA scaling factor - typically 2x rank
    lora_dropout=lora_dropout,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules="all-linear",  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
)


# Configure trainer
training_args = SFTConfig(
    output_dir="/root/autodl-tmp/sft/Qwen3-8B/sft-LoRA",
    max_steps=1000,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    logging_dir='/root/tf-logs',
    save_steps=100,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    bf16=True,
    warmup_steps=50
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["test"],
    processing_class=tokenizer,
    peft_config=peft_config
)

In [5]:
# 察看数据集处理结果
dataloader = trainer.get_train_dataloader()
batch = next(iter(dataloader))
print(tokenizer.decode(batch['input_ids'][0]))

<|im_start|>user
关键词抽取：
以手动换挡机构疲劳寿命试验为目的,构建了一种模拟驾驶员进行选档、换挡操作的试验平台,该平台集成二自由度运动滑台与气动加载装置为一体形成换挡运动加载机构.以该机构为研究对象,通过建立换挡与选挡的运动轨迹模型,分析换挡运动加载机构位移输出与运动轨迹之间的关系,通过分析换挡机构操纵杆受力情况,分别对换挡动作和选档动作进行力学分析,并得出加载力的计算方法.最后结合电气控制技术与气动控制技术,对系统进行了试验,结果表明,系统具有可行性与正确性.<|im_end|>
<|im_start|>assistant
<think>

</think>

换挡机构;试验平台;疲劳性能<|im_end|>
<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


In [6]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,2.247000,2.122826,2.143639,71046.000000,0.548373
200,1.885600,1.990344,1.974505,140655.000000,0.565260
300,2.143100,1.975090,1.961483,213877.000000,0.566731
400,2.009400,1.968589,2.000445,283989.000000,0.567496
500,2.072700,1.963211,1.974362,354307.000000,0.568844
600,1.914200,1.960248,1.978159,423796.000000,0.569066
700,2.091700,1.958303,1.971842,493956.000000,0.569176
800,2.005800,1.956370,1.960109,563521.000000,0.569459
900,2.014000,1.955541,1.975781,634604.000000,0.569344
1000,1.949300,1.954894,1.971816,705288.000000,0.569292


TrainOutput(global_step=1000, training_loss=2.111300054550171, metrics={'train_runtime': 647.3191, 'train_samples_per_second': 6.179, 'train_steps_per_second': 1.545, 'total_flos': 4.81544342034432e+16, 'train_loss': 2.111300054550171, 'epoch': 0.08080808080808081})

In [7]:
trainer.save_model('/root/autodl-tmp/sft/Qwen3-8B/sft-LoRA/best')

In [8]:
next(model.parameters()).dtype

torch.bfloat16